##**Coroutine that runs indefinitely until cancelled**

In [7]:
import asyncio
import time

async def continuous_ik_tracking_loop():
    """
    Runs an infinite 100Hz Inverse Kinematics loop until an explicit external
    cancellation command is received, executing a clean physical shutdown.
    """
    try:
        print("[IK Loop] Initializing tracking sensors... System Status: OK")
        cycle_count = 0

        while True:
            cycle_count += 1
            print(f"[IK Loop] Step {cycle_count}: Updating motor joint torques...")

            # This is our active checkpoint where cancellation is injected
            await asyncio.sleep(0.01)  # 100 Hz Frequency (10ms steps)

    except asyncio.CancelledError:
        print("\n[IK Loop] STOP COMMAND RECEIVED! Intercepting CancelledError...")
        print("[IK Loop] CLEANUP RUNNING: Safely reducing joint torque to 0.0 Nm...")
        print("[IK Loop] CLEANUP RUNNING: Locking brake systems... Homed safely.")
        raise  # Essential step to let the event loop know the task is finished

    finally:
        print("[IK Loop] Loop terminated. State Bridge cleared.")

async def test_cancellation():
    print("--- STARTING HARDWARE TRACKING TEST ---")
    # 1. Start the loop as a background task (Fire-and-Forget)
    tracking_task = asyncio.create_task(continuous_ik_tracking_loop())

    # 2. Let the robot track normally for 0.05 seconds (approx 5 steps)
    await asyncio.sleep(0.05)

    # 3. Trigger an Emergency Stop command!
    print("\n>>> EMERGENCY STOP SWITCH PRESSED! <<<")
    tracking_task.cancel()

    # 4. Wait a split-second to let the cleanup prints populate
    try:
        await tracking_task
    except asyncio.CancelledError:
        print("\n[Manager] Confirmation received: Task safely destroyed.")

# Run the test
await test_cancellation()

--- STARTING HARDWARE TRACKING TEST ---
[IK Loop] Initializing tracking sensors... System Status: OK
[IK Loop] Step 1: Updating motor joint torques...
[IK Loop] Step 2: Updating motor joint torques...
[IK Loop] Step 3: Updating motor joint torques...
[IK Loop] Step 4: Updating motor joint torques...
[IK Loop] Step 5: Updating motor joint torques...

>>> EMERGENCY STOP SWITCH PRESSED! <<<

[IK Loop] STOP COMMAND RECEIVED! Intercepting CancelledError...
[IK Loop] CLEANUP RUNNING: Safely reducing joint torque to 0.0 Nm...
[IK Loop] CLEANUP RUNNING: Locking brake systems... Homed safely.
[IK Loop] Loop terminated. State Bridge cleared.

[Manager] Confirmation received: Task safely destroyed.
